# JSON APIs for Digital Discourse — 2026

**DE0823 · Technical Week 4 · Google Colab · 90 minutes**

**Research question:** How can we retrieve structured online records, extract their text, and create a small reproducible discourse corpus?

**Previously:** In Week 3 we counted word forms in English and Latvian texts. **Today:** We learn *how structured data is represented*, retrieve it from an API, and reuse our word-counting techniques.

### Learning outcomes

By the end of this lesson you should be able to:

1. Explain why **JSON text is not the same thing as a Python dictionary**.
2. Translate JSON types into Python types **and** explain where reverse conversion loses information.
3. Distinguish **parsing / deserialization** from **serialization** and use `json.loads`, `json.load`, `json.dumps`, and `json.dump`.
4. Identify whether an API response is **text**, **bytes**, or **parsed Python data**.
5. Retrieve structured text records, keep provenance, count word forms, and export reproducible files.

**Timing:** JSON/Python concepts **15 min** · serialization and deserialization **15 min** · HTTP APIs **15 min** · discourse case study **20 min** · word frequencies **15 min** · exercises and interpretation **10 min**.

No keys, Drive mounts, or paid services are required. The notebook supports a clearly labeled **synthetic offline fallback**. Run cells in order.

## 1 · JSON text versus Python data structures (15 minutes)

### The essential distinction

**JSON (JavaScript Object Notation) is a language-independent text format used to exchange data.** Its contents follow a documented syntax: objects, arrays, strings, numbers, booleans, and `null`. Its name contains JavaScript, but it is not limited to JavaScript.

**Python dictionaries and lists are Python objects in memory.** They are containers your Python program can inspect and modify. A dictionary may represent the *same information* as a JSON object, but **the dictionary itself is not JSON text**.

A JSON document may be stored as a `.json` file, transmitted in an HTTP response, or held in a Python `str`. It becomes usable Python containers only *after parsing*. A `dict` becomes JSON text only *after serialization*.

| What you have | Example | Python type |
|---|---|---|
| Text containing a JSON document | `'{"name":"Anna","active":true}'` | `str` |
| Parsed data in Python memory | `{"name": "Anna", "active": True}` | `dict` |
| A file containing JSON text | `records.json` | File on disk (not a Python `dict`) |

**Watch the boolean:** JSON spells it `true` (lowercase); Python spells it `True` (capitalized). JSON uses `null`; Python uses `None`.

**Conceptual flow:**

```text
JSON document (characters / text)
             │ json.loads(text) — parse / deserialize
             ▼
Python objects (dict, list, str, int, ...)
             │ json.dumps(obj) — serialize
             ▼
JSON document (characters / text)
```

These are *representations* of information, not interchangeable Python types. When exchanging data over a network, the textual representation is typically encoded into bytes (often UTF-8).

### Compare the representations side by side

**JSON document** — valid JSON syntax:

```json
{
  "title": "Rīga",
  "active": true,
  "score": null,
  "tags": ["pilsēta", "kultūra"]
}
```

**Python dictionary literal** — valid Python syntax:

```python
{
    "title": "Rīga",
    "active": True,
    "score": None,
    "tags": ["pilsēta", "kultūra"]
}
```

The curly braces and nested structures look familiar in both, but **similar syntax does not make them the same object**. The first is a piece of text written according to JSON rules. The second is Python syntax that creates a dictionary.

JSON **also permits top-level arrays and simple values**, for example `[1, 2]`, `"hello"`, `42`, `true`, and `null`. Parsing JSON therefore does **not** always return a dictionary.

In [ ]:
import json
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import re
import unicodedata

# A Python string containing VALID JSON text.
json_text = '{"title": "Rīga", "active": true, "score": null, "tags": ["pilsēta", "kultūra"]}'
print('Before parsing:', type(json_text).__name__)

# Parsing produces a Python dictionary because the JSON root is an object.
parsed = json.loads(json_text)
print('After parsing:', type(parsed).__name__)
print('Nested value:', parsed['tags'][0], '| active:', parsed['active'])

# A native Python dictionary with equivalent contents.
record = {'title': 'Rīga', 'active': True, 'score': None, 'tags': ['pilsēta', 'kultūra']}
assert parsed == record
assert isinstance(json_text, str) and isinstance(parsed, dict)

### Translation table A — JSON → Python (parsing)

When Python parses JSON using the standard `json` module, its **default** type mappings are:

| JSON type | Python result | JSON example | Python example |
|---|---|---|---|
| **object** | `dict` | `{"x": 1}` | `{"x": 1}` |
| **array** | `list` | `[1, 2]` | `[1, 2]` |
| **string** | `str` | `"Rīga"` | `"Rīga"` |
| **number** (integer syntax) | `int` | `42` | `42` |
| **number** (decimal or exponent) | `float` | `3.14`, `1e3` | `3.14`, `1000.0` |
| **boolean** | `bool` | `true`, `false` | `True`, `False` |
| **null** | `NoneType` (value `None`) | `null` | `None` |

**Important:** JSON has one conceptual **number** type; the default Python parser chooses `int` or `float` based on its written form. JSON has no tuple, set, datetime, or integer-keyed object type.

### Translation table B — Python → JSON (serialization)

| Python input | JSON result | Important detail |
|---|---|---|
| `dict` | **object** | Keys in JSON objects are strings; Python allows other key types |
| `list` | **array** | List ordering preserved |
| `tuple` | **array** | The JSON form does **not** preserve tuple identity |
| `str` | **string** | Unicode text is supported |
| `int`, `float` | **number** | Non-finite floating values require care; valid JSON does not include NaN or Infinity |
| `True`, `False` | **boolean** | Written as `true`, `false` |
| `None` | **null** | Written as `null` |
| `set`, `datetime`, `bytes` | **not supported directly** | Require an explicit conversion / custom encoder |

Mappings are **not symmetrical**: `list` *and* `tuple` both become a JSON array, but a JSON array normally becomes a Python `list`. The default encoder may convert supported non-string dictionary keys to strings, changing their types.

These tables describe the *default Python* `json` library behavior. Other languages map the same JSON values to their own data types.

In [ ]:
samples = {
    'object': '{"x": 1}',
    'array': '[1, 2]',
    'string': '"Rīga"',
    'integer': '42',
    'decimal': '3.14',
    'exponent': '1e3',
    'boolean': 'true',
    'null': 'null',
}
for name, text_value in samples.items():
    value = json.loads(text_value)
    print(f'{name:9} JSON={text_value:14} -> Python {type(value).__name__:8} {value!r}')

assert isinstance(json.loads('42'), int)
assert isinstance(json.loads('1e3'), float)
assert json.loads('null') is None

### Parsing: how text becomes data

**Parsing** is analyzing text according to a language's syntax. For JSON, a parser recognizes quotes, braces, brackets, numbers, and JSON keywords. **Deserialization** is the wider operation of turning the serialized representation into in-memory objects. For this notebook, `json.loads()` performs both.

1. Receive JSON text (a Python `str`).
2. Check whether its characters satisfy JSON syntax.
3. Construct Python containers and values corresponding to its JSON elements.
4. Return the Python value; malformed JSON raises `json.JSONDecodeError`.

**Crucial:** The parser does not know your application-specific meaning. It can decode `{"age": "42"}`, but the value of `age` remains the string `"42"`, not the integer `42`. Likewise, a missing key is different from a key explicitly set to `null`.

**Do not use `eval()` to parse JSON.** Use `json.loads()`.

In [ ]:
age_record = json.loads('{"age": "42", "verified": false, "note": null}')
print('age:', age_record['age'], '| type:', type(age_record['age']).__name__)
print('explicit null:', age_record['note'] is None)
print('missing field:', 'author' not in age_record)

# Malformed JSON is deliberately handled, so Run all continues.
invalid_json = "{'active': True}"  # Looks like a Python dict; NOT JSON.
try:
    json.loads(invalid_json)
except json.JSONDecodeError as exc:
    print('Expected parsing error:', exc.msg)
else:
    raise AssertionError('Invalid JSON was unexpectedly accepted')

### Navigating nested structures and missing fields

JSON documents often contain objects inside arrays inside other objects. After parsing, use Python's normal indexing and dictionary operations:

- `obj['results']` follows an **object key** (dictionary).
- `obj['results'][0]` selects an **array element** (list; indexing starts at zero).
- `obj.get('author')` retrieves an optional key or returns `None` if absent.

**Remember:** A missing key and an explicit JSON `null` are different data states. Neither should automatically be interpreted as an empty string or zero.

In [ ]:
example = {'results': [{'id': 'a1', 'metadata': {'language': 'lv'}, 'text': 'Vārdi veido diskursu.'}]}
print(example.keys())
print(example['results'][0]['metadata'].get('language'))
print(example['results'][0].get('author'))  # unknown, not an empty author
try:
    print(example['results'][0]['missing'])
except KeyError as exc:
    print('Expected missing-key example:', exc)

## 2 · Serialization and deserialization (15 minutes)

### Why convert between formats?

Two programs cannot exchange a live Python dictionary directly as a universal data format: the dictionary is an object belonging to one running Python process. They can exchange a standardized **representation** of its data, such as JSON text encoded into bytes for transmission.

- **Serialization / encoding:** Python object → JSON text (and optionally a file or bytes on a network).
- **Deserialization / decoding:** JSON text → Python object.
- **Parsing:** checking and interpreting the JSON syntax during deserialization.

This makes JSON useful for API responses, configurations, datasets, and data exchange among Python, JavaScript, R, Java, and many other languages.

### Why `str(my_dict)` is not JSON serialization

`str()` creates a human-readable Python representation, which may use single quotes, `True`, and `None`. JSON requires double-quoted strings and its own keyword spelling. **Use `json.dumps()`, not `str()`, to produce JSON.**

In [ ]:
python_data = {'city': 'Rīga', 'active': True, 'population_estimate': None}
python_representation = str(python_data)
valid_json_text = json.dumps(python_data, ensure_ascii=False)

print('str(dict):', python_representation)
print('json.dumps(dict):', valid_json_text)
print('Both results are Python strings:', type(python_representation).__name__, type(valid_json_text).__name__)

assert json.loads(valid_json_text) == python_data
try:
    json.loads(python_representation)
except json.JSONDecodeError:
    print('As expected, the Python dict representation is not valid JSON')
else:
    raise AssertionError('Expected the Python representation to be invalid JSON')

### The four `json` functions — strings versus files

| Function | What it accepts | What it does | Result |
|---|---|---|---|
| `json.loads(text)` | JSON text, usually `str` | **Deserializes** text | Returns a Python object |
| `json.load(file)` | Readable file-like stream | **Deserializes** the stream's JSON | Returns a Python object |
| `json.dumps(obj)` | Supported Python object | **Serializes** to text | Returns a Python `str` |
| `json.dump(obj, file)` | Python object + writable stream | **Serializes** into the stream | Writes JSON; returns `None` |

The final **s** in `loads` and `dumps` is a handy mnemonic for **string**. A `dump` function does not necessarily write to disk: it writes to a *stream*, which might be an open file or an in-memory text buffer.

```text
Python object -- json.dumps() --> JSON string
Python object -- json.dump()  --> JSON written into a stream

JSON string   -- json.loads() --> Python object
JSON stream   -- json.load()  --> Python object
```

A `.json` file is still a **text file**. Reading its raw contents with `file.read()` gives text; calling `json.load(file)` gives Python data.

The serializer typically produces JSON characters, whereas HTTP transport sends **bytes**. UTF-8 is the standard encoding for interoperable JSON exchanged over networks.

In [ ]:
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)
example_path = output_dir / 'example_record.json'

# Python dict -> JSON text written to a UTF-8 file
with example_path.open('w', encoding='utf-8') as f:
    write_result = json.dump(record, f, ensure_ascii=False, indent=2)
assert write_result is None

# Raw file reading returns a string, not a dictionary.
with example_path.open('r', encoding='utf-8') as f:
    raw_file_text = f.read()
print('file.read() gives:', type(raw_file_text).__name__)

# Parsing directly from a file gives a Python dictionary.
with example_path.open('r', encoding='utf-8') as f:
    restored = json.load(f)
print('json.load(file) gives:', type(restored).__name__)

# Parsing JSON text gives an equivalent Python dictionary.
assert json.loads(raw_file_text) == restored == record
print('Saved:', example_path)
print(raw_file_text)

### Round trips: JSON does not preserve every Python type

A **round trip** is Python → JSON → Python. It preserves the information represented by JSON, but may **not reconstruct the original Python types**:

- A Python `tuple` is serialized as a JSON array and read back as a `list`.
- A dictionary key such as integer `1` becomes JSON object key `"1"` and returns as a string key.
- A `set` or `datetime` cannot be serialized directly with the default encoder; you must choose an explicit representation.
- Even valid JSON conversion does not guarantee that every application-specific meaning survives.

For research data, **decide which type information and metadata must survive** before exchanging or archiving your dataset.

In [ ]:
original = {'coordinates': (56.95, 24.11)}
round_trip = json.loads(json.dumps(original))
print('Before:', type(original['coordinates']).__name__, original)
print('After: ', type(round_trip['coordinates']).__name__, round_trip)
assert isinstance(original['coordinates'], tuple)
assert isinstance(round_trip['coordinates'], list)
assert original != round_trip

with_integer_key = {1: 'one'}
restored_keys = json.loads(json.dumps(with_integer_key))
print('Integer key before:', with_integer_key, '| string key after:', restored_keys)
assert restored_keys == {'1': 'one'}

try:
    json.dumps({'tags': {'history', 'language'}})  # set is unsupported
except TypeError as exc:
    print('Expected unsupported-type error:', type(exc).__name__)
else:
    raise AssertionError('A set should not serialize with the default encoder')

### Unicode, Latvian text, and file encoding

JSON strings support Unicode. However, **displaying Latvian letters directly** and **using UTF-8 to store bytes** are separate choices:

- `json.dumps(data)` defaults to `ensure_ascii=True`: `ā` may appear as an escape such as `\u0101`. **This is still valid JSON and decodes to the same character.**
- `json.dumps(data, ensure_ascii=False)` keeps characters such as `ā`, `ģ`, and `ņ` visible in the text.
- `open(path, encoding='utf-8')` specifies the character encoding when saving or loading a file.

This is especially important when working with Latvian texts. Do **not** confuse JSON character escaping with whether an underlying file is encoded correctly.

In [ ]:
latvian = {'city': 'Rīga', 'topic': 'vēsture un kultūra', 'question': 'Kā?'}
escaped = json.dumps(latvian)               # ASCII escapes in JSON text
readable = json.dumps(latvian, ensure_ascii=False)
print('JSON with escapes:   ', escaped)
print('JSON readable:       ', readable)
assert json.loads(escaped) == json.loads(readable) == latvian
assert '\\u' in escaped and 'Rīga' in readable

### Check your understanding — predict before running

1. What type does `json.loads('[1, 2, 3]')` return?
2. What does `json.loads('null')` return? Is it an empty dictionary?
3. Which of the four functions returns a **string**, and which writes to a **file-like stream**?
4. Why is `str({'ok': True})` generally **not valid JSON**?
5. Can JSON distinguish a Python tuple from a Python list on a round trip?
6. Is `json.loads('{"age": "42"}')['age']` an integer?
7. What does `ensure_ascii=False` change — and what does it **not** change?

**Instructor answers (reveal after discussion):** (1) `list`; (2) `None`, not `{}`; (3) `dumps` returns a string, `dump` writes; (4) Python syntax differs (quotes and booleans); (5) no; (6) no, it is `str`; (7) JSON text escaping/display, not the file's encoding.

**Optional pair task:** Write your own nested Python `dict`, serialize it, parse it back, and inspect `type()` at each step.

In [ ]:
assert isinstance(json.loads('[1, 2, 3]'), list)
assert json.loads('null') is None
assert json.dumps({'ok': True}) == '{"ok": true}'
assert isinstance(json.dumps(record), str)
assert isinstance(json.loads('{"age": "42"}')['age'], str)
print('Concept checks passed')

### Bridge to APIs: `response.text` versus `response.json()`

Once a server sends a JSON response, the HTTP library gives us a **response object** — not yet a Python dictionary.

- `response.content` provides the **response body as bytes**.
- `response.text` provides the **response body decoded into a Python string**. If the body contains JSON, it is *still a string*.
- `response.json()` attempts to **parse the response body as JSON** and returns the resulting Python object (often a `dict` or `list`, but other JSON types are possible). Despite its name, it does **not** return a JSON string.

```text
HTTP bytes -> decoded response.text (str) -> JSON parser -> Python dict/list/...
                          response.json() parses the body directly
```

**Distinguish the checks:** An HTTP 200 response does not guarantee valid JSON; valid JSON does not guarantee the structure your code expects; and valid structure does not guarantee appropriate research data.

## 3 · HTTP APIs and defensive retrieval (15 minutes)

An API is an interface to a service. A GET request asks for a resource. HTTP 200 means success at the HTTP level, **not** that the response is valid JSON or has your expected structure.

Use an explicit timeout, check the HTTP status, parse JSON, and validate its schema. The initial API demonstration uses [JSONPlaceholder](https://jsonplaceholder.typicode.com/), which returns **synthetic records for practice**—not empirical discourse evidence.

In [ ]:
import requests

def fetch_json(url, *, params=None, timeout=12):
    '''Return parsed JSON; raise on network, HTTP, or JSON errors.'''
    response = requests.get(url, params=params, timeout=timeout, headers={'Accept': 'application/json', 'User-Agent': 'DE0823-teaching-notebook/2026'})
    response.raise_for_status()
    return response.json()

print('requests version:', requests.__version__)

In [ ]:
url = 'https://jsonplaceholder.typicode.com/users'
try:
    response = requests.get(url, timeout=12, headers={'Accept': 'application/json', 'User-Agent': 'DE0823-teaching-notebook/2026'})
    response.raise_for_status()
    print('response.content:', type(response.content).__name__)  # bytes
    print('response.text:', type(response.text).__name__)        # str
    demo_users = response.json()                                # parsed Python value
    assert isinstance(demo_users, list) and demo_users
    print('response.json():', type(demo_users).__name__)         # list for this endpoint
    print('Users:', len(demo_users), '| first:', demo_users[0].get('name'))
    print('Nested city:', demo_users[0].get('address', {}).get('city'))
except (requests.RequestException, ValueError, AssertionError) as exc:
    demo_users = []
    print('Live introductory API unavailable; continue with the local corpus:', type(exc).__name__)

## 4 · Digital discourse case study (20 minutes)

Use the **MediaWiki REST API** for short article summaries in two languages. This is encyclopedic discourse, **not** social-media discourse. We request a small, explicitly chosen set of articles; it is a convenience sample, not a representative corpus.

**Reliability rule:** The notebook has an embedded synthetic fallback with explicit provenance. If the network or endpoint fails, the analysis still runs, but the outputs must not be presented as retrieved Wikipedia content. API payload fields may change; validate before using them.

In [ ]:
PAGES = [('en', 'Digital_humanities'), ('en', 'Discourse_analysis'), ('lv', 'Rīga'), ('lv', 'Latviešu_valoda')]
# Clearly synthetic offline fixture, not quotations or actual Wikipedia extracts.
OFFLINE_FIXTURE = [
    {'id':'fixture-en-1','language':'en','title':'Digital humanities','text':'Digital humanities studies digital sources and research methods. Researchers examine texts and metadata.'},
    {'id':'fixture-en-2','language':'en','title':'Discourse analysis','text':'Discourse analysis examines language and context. Texts can be compared using careful methods.'},
    {'id':'fixture-lv-1','language':'lv','title':'Rīga','text':'Rīga ir Latvijas pilsēta. Pilsētas vēsture un kultūra ir pētniecības tēmas.'},
    {'id':'fixture-lv-2','language':'lv','title':'Latviešu valoda','text':'Latviešu valoda ir pētniecības objekts. Valoda un teksts palīdz analizēt diskursu.'},
]

def extract_summary(payload, language, slug):
    if not isinstance(payload, dict):
        raise ValueError('Expected JSON object')
    text = payload.get('extract')
    title = payload.get('title')
    if not isinstance(text, str) or not text.strip() or not isinstance(title, str):
        raise ValueError('Unexpected summary schema')
    return {'id': f'{language}:{slug}', 'language': language, 'title': title, 'text': text.strip()}

assert len(OFFLINE_FIXTURE) == 4

In [ ]:
from urllib.parse import quote

USE_LIVE_API = True  # set False for guaranteed offline demonstration
retrieved_at = datetime.now(timezone.utc).isoformat()
records = []
source_urls = []
source_kind = 'synthetic_fixture'
if USE_LIVE_API:
    try:
        for lang, slug in PAGES:
            url = f'https://{lang}.wikipedia.org/api/rest_v1/page/summary/{quote(slug)}'
            payload = fetch_json(url)
            records.append(extract_summary(payload, lang, slug))
            source_urls.append(url)
        source_kind = 'mediawiki_live'
    except (requests.RequestException, ValueError, KeyError) as exc:
        print('Live API failed; using synthetic teaching fixture:', type(exc).__name__)
        records = []
        source_urls = []
if not records:
    records = [dict(item) for item in OFFLINE_FIXTURE]
    source_urls = ['embedded:OFFLINE_FIXTURE']
print('Source:', source_kind, '| records:', len(records))
for item in records:
    print(item['language'], item['title'], '—', item['text'][:90])

### Provenance and limitations

Record retrieval time and exact source endpoints. The retrieval time is *not* the article publication date. API data can change. A fallback fixture is pedagogical test data and must not be cited as real-world evidence. Language-specific word-form counts are not directly comparable without considering morphology and corpus size.

In [ ]:
corpus = {'metadata': {'course': 'DE0823', 'retrieved_at_utc': retrieved_at, 'source_kind': source_kind, 'source_urls': source_urls, 'selection': 'four preselected page titles; convenience sample', 'extraction': 'MediaWiki summary extract field or synthetic teaching text', 'limitations': 'Not representative; summary text only; synthetic fallback is not empirical evidence'}, 'records': records}
assert all(isinstance(r.get('text'), str) and r['text'].strip() and r.get('language') in ('en','lv') for r in corpus['records'])
corpus_path = output_dir / 'discourse_corpus.json'
with corpus_path.open('w', encoding='utf-8') as f:
    json.dump(corpus, f, ensure_ascii=False, indent=2)
print('Saved:', corpus_path)

## 5 · From JSON to word frequencies (15 minutes)

Reuse the Week 3 idea: Unicode normalization, case folding, tokenization, and `Counter`. This simple tokenizer counts surface word forms, not lemmas. For Latvian, inflection creates many forms of the same lexeme. Counts across languages should be interpreted separately.

In [ ]:
def tokenize(text):
    normalized = unicodedata.normalize('NFC', text).casefold()
    # Extract Unicode letter sequences, avoiding digits and underscores.
    return re.findall(r'[^\W\d_]+(?:[\'’\-][^\W\d_]+)*', normalized, flags=re.UNICODE)

frequencies = {}
for lang in sorted({r['language'] for r in records}):
    tokens = [token for r in records if r['language'] == lang for token in tokenize(r['text'])]
    frequencies[lang] = Counter(tokens)
    print(lang, '| tokens:', len(tokens), '| distinct word forms:', len(frequencies[lang]), '| top:', frequencies[lang].most_common(8))
assert all(sum(c.values()) > 0 for c in frequencies.values())

In [ ]:
import csv
frequency_path = output_dir / 'word_frequencies.tsv'
with frequency_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f, delimiter='\t')
    writer.writerow(['language', 'word_form', 'count'])
    for lang, counts in sorted(frequencies.items()):
        for word, count in sorted(counts.items(), key=lambda pair: (-pair[1], pair[0])):
            writer.writerow([lang, word, count])
with frequency_path.open(encoding='utf-8', newline='') as f:
    rows = list(csv.DictReader(f, delimiter='\t'))
assert rows and set(rows[0]) == {'language','word_form','count'}
assert sum(int(r['count']) for r in rows) == sum(sum(c.values()) for c in frequencies.values())
print('Saved:', frequency_path, '| rows:', len(rows))

## 6 · Student exercises and interpretation (10 minutes)

**Exercise A (5 minutes):** Change `USE_LIVE_API` to `False`, run from the retrieval cell onward, and inspect the provenance fields in `outputs/discourse_corpus.json`. Explain why the source label changes.

**Exercise B (5 minutes):** Identify the three most frequent word forms in each language. Explain why these results cannot establish which topics dominate English-language versus Latvian-language discourse.

**Optional extension:** Add one article title to `PAGES` and rerun. What happens if the title does not exist? How would you handle partial success without mixing datasets silently?

**Reflection:** What is lost when a document is reduced to its summary text and then to a bag of word forms? Consider genre, authorship, context, selection bias, tokenization, and source revisions.

In [ ]:
print('Final validation')
with corpus_path.open(encoding='utf-8') as f:
    saved = json.load(f)
assert saved['metadata']['source_kind'] in {'mediawiki_live','synthetic_fixture'}
assert len(saved['records']) == len(records)
assert all(r['id'] and r['title'] for r in saved['records'])
print('OK: reproducible corpus JSON and TSV created in outputs/')

## Further reading and instructor notes

- [Python's `json` library documentation](https://docs.python.org/3/library/json.html) — type mapping and encoder/decoder behavior
- [RFC 8259: The JavaScript Object Notation (JSON) Data Interchange Format](https://www.rfc-editor.org/rfc/rfc8259) — JSON syntax and interoperability
- [Requests quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/) — `Response.text`, `Response.json()`, and status handling
- [MediaWiki API documentation](https://www.mediawiki.org/wiki/API:Main_page)
- [Week 3: CountingWords_2026](../TextProcessing/CountingWords_2026.ipynb)

### Notes for the live instructor

- Ask *“What is the Python type?”* every time you cross a representation boundary; use `type()` instead of relying on visual similarity.
- Start with the side-by-side JSON and Python literals; explicitly contrast `true/True`, `null/None`, and JSON double quotes.
- Read the two translation tables in **both directions**, demonstrating tuple → array → list and integer-key → string-key.
- Ask students to predict outcomes *before* executing the check cells.
- Stress that `json.dumps()` produces a `str`, not a file, and `json.load()` consumes an open stream.
- The JSON sections are conceptually dense: prioritize those demonstrations over optional API variants if class discussion takes longer.
- Run the notebook with `USE_LIVE_API=False` to verify fully offline operation, then use live mode if the network permits.
- Wikipedia summaries are a **convenience sample of encyclopedic discourse**, not a representative social-media corpus. The fallback is synthetic and must never be reported as retrieved Wikipedia data.
- Notebook outputs live in the temporary Colab filesystem and require downloading if they should persist.

**Next topic:** web scraping and pandas.